<a href="https://colab.research.google.com/github/gigimcc4/IBM_for_researchers/blob/main/Tutorial_3_IBM_Granite_Docling_RAG_(shared).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **IMPORTANT — FIRST — MAKE A COPY OF THIS NOTEBOOK, OR NOTHING WILL BE SAVED!**
## **FILE → SAVE A COPY IN DRIVE / GITHUB**

# Tutorial 3: IBM Granite + Docling — Ask Questions About Your Own Documents

*Created for the Nagoya University 5-Day Faculty Seminar on Teaching with AI*  
**Dr. Jeanne McClure, PhD** | NC State University Data Science Academy / Ars Innovate

---

### 📜 License

© 2026 Dr. Jeanne McClure. This work is licensed under a [Creative Commons Attribution-NonCommercial 4.0 International License (CC BY-NC 4.0)](https://creativecommons.org/licenses/by-nc/4.0/).

This material was in part developed with support from the National Science Foundation, DUE-2313644. All opinions, findings, conclusions and recommendations expressed herein are those of the authors and do not necessarily reflect the views of the Foundation.

---

**What this tutorial is:** A practical walkthrough of building a working RAG system on **your own PDF documents** — research papers, syllabi, reports, anything. You'll use two IBM open-source tools:

- **Docling** — IBM's document parser that extracts clean text from PDFs, even messy multi-column academic papers with tables
- **IBM Granite 3.1 2B** — The same open-source LLM from Tutorial 1, now serving as the answer-generation step in a full RAG pipeline

**What you'll be able to do after this tutorial:**
- Point Docling at a folder of PDFs and get clean, structured text out
- Understand — really understand — how vector embeddings make semantic search work
- Build a searchable index of your own documents
- Ask questions in plain English and get answers with source citations

**How this connects to Tutorial 2:**  
Tutorial 2 showed you the RAG pipeline conceptually using small fake documents.  
This tutorial runs the *same pipeline* on your **real documents** using IBM tools built for research use.

**Time:** ~60–75 minutes  
**Requires:** Tutorial 1 setup familiarity (GPU, Colab basics)

## ✅ What You Need

| # | Requirement | Notes |
|---|---|---|
| 1 | **A Google account** | To open Colab and mount your Drive |
| 2 | **PDFs in Google Drive** | Research papers, syllabi, reports — 2–5 PDFs to start |
| 3 | **T4 GPU enabled in Colab** | Runtime → Change runtime type → T4 GPU |

### What You Do NOT Need

| ❌ Not needed | Why not? |
|---|---|
| IBM account or API key | Granite and Docling are both fully open source |
| Perfect PDFs | Docling handles messy layouts, tables, and multi-column formats |
| Prior Python experience | Every cell is explained — just press Shift+Enter |

> **One thing to prepare:** Before running this notebook, identify a folder in your Google Drive containing 2–5 PDFs you'd like to query. Research papers work well. Scanned image-only PDFs will not work — Docling needs text-based PDFs.

## 📚 Resource Links

You don't need to read any of these to complete the tutorial — they're here for going deeper.

### About IBM Docling
- [Docling GitHub](https://github.com/DS4SD/docling) — Source code, issues, and community — this is where the project lives
- [Docling on Hugging Face](https://huggingface.co/docling-project) — The layout detection models Docling uses under the hood
- [IBM Research Blog: Docling](https://www.docling.ai/) — IBM's announcement explaining why they built it and what problems it solves
- [Docling Technical Paper](https://arxiv.org/abs/2408.09869) — The research paper describing Docling's architecture and benchmarks against other PDF parsers

### About IBM Granite
- [IBM Granite Official Page](https://www.ibm.com/granite) — Overview of the Granite model family, use cases, and IBM's responsible AI commitments
- [Granite Models on Hugging Face](https://huggingface.co/ibm-granite) — All available Granite models; browse sizes and task-specific variants
- [IBM Granite GitHub](https://github.com/ibm-granite) — Source code, recipes, and community resources
- [IBM Granite Cookbook](https://github.com/ibm-granite-community/granite-kitchen) — Practical use-case examples including RAG pipelines


### About RAG and Vector Embeddings
- [IBM: What is RAG?](https://www.ibm.com/think/topics/retrieval-augmented-generation) — IBM's plain-language explainer on retrieval-augmented generation
- [Sentence Transformers Documentation](https://www.sbert.net/) — The embedding library we use; includes model comparisons and use-case guides
- [ChromaDB Documentation](https://docs.trychroma.com/) — The vector database we use for storing and searching embeddings
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) — A visual explanation of how language models work (no math required)

### For SAS/Stata Users
- [Python for SAS Users](https://pandas.pydata.org/docs/getting_started/comparison/comparison_with_sas.html) — Official side-by-side comparison
- [Python for Stata Users](https://pandas.pydata.org/docs/getting_started/comparison/comparison_with_stata.html) — Official side-by-side comparison

## 📄 What is Docling?

Docling is IBM Research's open-source document parser, released in 2024. It solves a problem that has frustrated researchers for years: **PDFs are terrible data sources.**

A PDF was designed for *printing*, not for *reading by software*. When a basic PDF reader extracts text, you often get:
- Two-column layouts merged into one scrambled stream
- Table cells extracted in the wrong order
- Headers and footers mixed into paragraph text
- Equations and captions replaced with nothing

Docling understands document *structure*. It uses a layout detection model (similar to a computer vision system) to identify what each region of a page actually *is* — a heading, a paragraph, a table, a figure caption — before extracting the text.

### Docling vs. a basic PDF reader

| | Basic PDF reader (`pymupdf`, `pdfminer`) | Docling |
|---|---|---|
| Two-column layouts | Merged into random order | Reads left column, then right |
| Tables | Flattened or scrambled | Preserved as structured markdown |
| Reading order | Often wrong | Layout-aware, correct order |
| Headings | Treated as regular text | Identified and marked |
| Speed | Very fast | Slower (~30–60 sec/page) — runs a vision model |
| Best for | Simple single-column text | Academic papers, reports, slides |

### Why IBM built Docling

IBM Research built Docling specifically for **enterprise document workflows** — situations where accuracy matters more than speed. For a faculty lit review pipeline or a department document search system, that trade-off is correct: you run the parser once, and the resulting clean text powers every search forever.

> **Docling is open source under the MIT license** — free to use for research, teaching, and institutional tools, with no API calls or cloud dependency required.

🔗 [Docling GitHub](https://github.com/DS4SD/docling) · [Docling Docs](https://ds4sd.github.io/docling/) · [IBM Research Blog Post](https://research.ibm.com/blog/docling)

## 🏗️ What We're Building

```
YOUR GOOGLE DRIVE FOLDER
  └── paper1.pdf          ┐
  └── paper2.pdf          ├──▶  DOCLING  (parse + extract clean text)
  └── paper3.pdf          ┘
                                       │
                                       ▼
                             CHUNK  (split into ~400-word pieces)
                                       │
                                       ▼
                             EMBED  (convert meaning to numbers)
                                       │
                                       ▼
                             STORE  (ChromaDB vector index)
                                       │
               ┌───────────────────────┘
               │
  YOUR QUESTION  ──▶  RETRIEVE most relevant chunks
                                       │
                                       ▼
                             IBM GRANITE generates answer
                             with citations
```

This is the full RAG pipeline from Tutorial 2, but now:
- **Ingest** uses Docling (IBM) instead of hardcoded fake text
- **Generate** uses Granite (IBM) instead of just printing the prompt
- **Documents** come from your real Google Drive folder

We'll pause at the **Embed** step for a deeper look — this is the concept most people find surprising, and understanding it changes how you design document collections for students.

---
# ⚙️ Setup

Four steps. Run them in order and wait for the ✅ before moving to the next one.

### Step 1: Check GPU

Granite needs a GPU. If this cell returns ❌, go to **Runtime → Change runtime type → T4 GPU** and re-run. We did this yesterday to change runtime from CPU to GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                        '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ GPU available: {result.stdout.strip()}')
    print('   Granite and Docling will both use this.')
else:
    print('❌ No GPU detected.')
    print('   Go to: Runtime → Change runtime type → T4 GPU → Save')
    print('   Then re-run this cell.')


### Step 2: Install Libraries

| Library | What it does | Source |
|---|---|---|
| `docling` | Parses PDFs into clean structured text | IBM Research |
| `transformers` | Loads and runs IBM Granite | Hugging Face |
| `sentence-transformers` | Converts text to meaning vectors | Hugging Face |
| `chromadb` | Stores and searches those vectors | Open source |
| `accelerate` | GPU memory management for Granite | Hugging Face |

This takes **2–4 minutes**. Wait for the ✅.

In [ ]:
!pip install -q docling transformers accelerate torch sentencepiece
!pip install -q sentence-transformers chromadb scikit-learn

print('✅ All libraries installed!')
print('   docling              — PDF parsing (IBM Research)')
print('   transformers + torch — IBM Granite model runtime')
print('   sentence-transformers — text → meaning vectors')
print('   chromadb             — vector search database')
print('   scikit-learn         — used for embedding visualization')


### Step 3: Mount Google Drive

You'll see a permissions dialog — click **Connect to Google Drive** and sign in.

> Colab accesses your Drive files but does not store or transmit them anywhere. The connection exists only for this session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')


### Step 4: Point to Your PDF Folder

Change `FOLDER_PATH` to the folder in your Google Drive containing your PDFs.

**How to find your path:**
1. Open Google Drive in a browser tab
2. Navigate to your folder
3. Copy the folder names exactly as shown (case-sensitive)
4. Build the path as: `/content/drive/MyDrive/FolderName/SubfolderName`

**Examples:**
```
'/content/drive/MyDrive/Research Papers'
'/content/drive/MyDrive/Course Materials/Unit 3'
'/content/drive/MyDrive/Lit Review/Climate'
```

> Note: Colab uses `MyDrive` (no space) — not `My Drive`

In [ ]:
import os

# ── CHANGE THIS to your own folder ──────────────────────────────────────────
FOLDER_PATH = '/content/drive/MyDrive/NAGOYA/Nagoya_Architects/90_Minute_Conceptuals/Day_1/notebookLM_activity'

# ── Folder check ─────────────────────────────────────────────────────────────
if not os.path.exists(FOLDER_PATH):
    print('❌ Folder not found.')
    print(f'   Path tried: {FOLDER_PATH}')
    print('   Common fixes:')
    print('   - Use MyDrive (no space), not My Drive')
    print('   - Folder names are case-sensitive')
    print('   - Make sure Drive is mounted (run Step 3 first)')
else:
    pdf_files = [f for f in os.listdir(FOLDER_PATH) if f.lower().endswith('.pdf')]
    print(f'✅ Folder found.')
    print(f'   PDFs detected: {len(pdf_files)}')
    for f in pdf_files:
        size_kb = os.path.getsize(os.path.join(FOLDER_PATH, f)) // 1024
        print(f'   📄 {f}  ({size_kb:,} KB)')
    if len(pdf_files) == 0:
        print('   ⚠️  No PDFs found. Check that files end in .pdf (lowercase)')


---
# 📄 Step 1: Parse PDFs with Docling

This is where Docling earns its place. We feed it every PDF in your folder and get back clean, structured text — with page numbers, section headings, and tables preserved.

### What Docling does behind the scenes

```
PDF page (image-like)  →  Layout detection model  →  Regions identified
                                                         │
                            ┌────────────────────────────┤
                            │                            │
                          heading                    body text
                          table                      figure caption
                            │
                            ▼
                     Clean markdown text
                     in correct reading order
```

**Why this matters for your RAG pipeline:**  
Garbage in = garbage out. If the parser scrambles a two-column paper, the chunks will be scrambled, the embeddings will be confused, and the answers will be wrong. Docling's layout awareness means the text you feed the pipeline is as clean as possible before any AI sees it.

> ⏱️ **Expect ~30–60 seconds per PDF.** Docling is running a vision model on each page — this is much slower than a basic text extractor, but the quality difference is significant for academic papers.

🔗 [Docling GitHub](https://github.com/DS4SD/docling) · [Docling Technical Paper](https://arxiv.org/abs/2408.09869)

In [ ]:
from docling.document_converter import DocumentConverter
from pathlib import Path

# Initialise Docling
# DocumentConverter loads the layout detection model on first use
print('Initialising Docling (loads layout model on first run)...')
converter = DocumentConverter()
print('✅ Docling ready\n')

parsed_documents = {}  # {filename: extracted_text}

pdf_files = [f for f in os.listdir(FOLDER_PATH) if f.lower().endswith('.pdf')]

if not pdf_files:
    print('⚠️  No PDFs found. Check your FOLDER_PATH above.')
else:
    print(f'Parsing {len(pdf_files)} PDF(s) with Docling...')
    print('(~30–60 seconds per file)\n')

    for filename in pdf_files:
        filepath = os.path.join(FOLDER_PATH, filename)
        print(f'  📄 Parsing: {filename}...')
        try:
            result = converter.convert(filepath)
            text = result.document.export_to_markdown()
            parsed_documents[filename] = text
            word_count = len(text.split())
            print(f'     ✅ {word_count:,} words extracted')
        except Exception as e:
            print(f'     ❌ Could not parse: {e}')

    print(f'\n✅ Parsed {len(parsed_documents)} of {len(pdf_files)} PDFs.')


### 🔍 Inspect the Parsed Output

Let's check what Docling extracted. **What to look for:**
- Is the text readable and in the right order?
- Are headings marked with `#` symbols (markdown format)?
- Are tables preserved with `|` column separators?

If a document looks garbled or nearly empty, it may be a scanned image PDF — those require OCR and are not supported in this version.

In [ ]:
for filename, text in parsed_documents.items():
    print(f'{'='*65}')
    print(f'📄 {filename}')
    print(f'   {len(text):,} characters  |  {len(text.split()):,} words')
    print(f'{'─'*65}')
    print(text[:1000])
    if len(text) > 1000:
        print('\n... [truncated — full text is in parsed_documents[filename]]')
    print()


---
# ✂️ Step 2: Chunk the Documents

As you saw in Tutorial 2, we split each document into smaller **chunks** that can be searched and cited individually.

**Why we can't just feed the whole paper to the AI:**  
Language models have a maximum input size (a "context window"). A 15-page paper might have 6,000 words — well over what the retrieval step can compare efficiently. More importantly, if we fed the *entire* paper every time, the model would get everything — including all the irrelevant sections — and the answer quality would drop.

**Our chunking strategy:**
- Target size: ~400 words per chunk
- Overlap: 50 words between adjacent chunks — so ideas that span a boundary don't get cut in half
- Each chunk carries metadata: which file it came from, which chunk number

| Chunk too small (~50 words) | Chunk too large (~2,000 words) |
|---|---|
| Loses context — a number without its label | Buries the answer in irrelevant text |
| Very precise citations | Vague citations |
| Like highlighting a single word | Like highlighting the whole page |

In [ ]:
def chunk_text(text, chunk_size=400, overlap=50):
    """
    Split text into overlapping word-based chunks.
    chunk_size : target words per chunk
    overlap    : words shared between adjacent chunks
    """
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(' '.join(words[start:end]))
        if end == len(words):
            break
        start += chunk_size - overlap
    return chunks

all_chunks, all_metadata, all_ids = [], [], []
chunk_id = 0

for filename, text in parsed_documents.items():
    doc_chunks = chunk_text(text)
    for i, chunk in enumerate(doc_chunks):
        all_chunks.append(chunk)
        all_metadata.append({'source': filename, 'chunk_index': i,
                             'total_chunks': len(doc_chunks)})
        all_ids.append(f'chunk_{chunk_id}')
        chunk_id += 1

print(f'✅ Chunking complete.')
print(f'   Documents : {len(parsed_documents)}')
print(f'   Total chunks : {len(all_chunks)}')
print()
for filename in parsed_documents:
    n = len([m for m in all_metadata if m['source'] == filename])
    print(f'   {filename}: {n} chunks')


---
# 🔢 Step 3: Vector Embeddings — The Key Concept

This is the step that makes RAG *work* — and the one that surprises people most. Take a few minutes here before running the code.

## What is a vector embedding?

An embedding is the answer to a simple but hard question:  
**How do you compare the *meaning* of two pieces of text?**

Computers are very good at comparing numbers. They are not good at comparing the meaning of sentences. The solution: **convert text into numbers that capture meaning**, so that texts with similar meanings get similar numbers.

### The key idea in one line:

> An embedding model reads a piece of text and produces a list of numbers — a **vector** — where texts that mean similar things have similar vectors.

---

## The Analogy: A Map of Meaning

Imagine you had to physically place every sentence you've ever read on a giant map, where sentences about similar topics are placed close together:

```
                              MAP OF MEANING
   
   BIOLOGY                              FINANCE
   ·  'cells divide by mitosis'         · 'interest rates affect bonds'
   ·  'DNA replication occurs'          · 'stock market volatility'
   ·  'proteins fold into shapes'       · 'quarterly earnings report'
   
                   EDUCATION  (somewhere in the middle)
                   · 'students learn better with practice'
                   · 'active learning improves retention'
                   · 'exam scores correlate with attendance'
```

When you ask a question, it also gets placed on this map.  
The system finds the sentences **nearest to your question on the map** — those are the chunks it retrieves.

**This is why semantic search works differently from keyword search:**

| Keyword search | Semantic/vector search |
|---|---|
| Finds exact word matches | Finds similar *meaning*, not just matching words |
| `"learning outcomes"` misses `"what students achieved"` | Both map to nearly the same location |
| Fast and exact | Slower but concept-aware |

---

## What is a vector, really?

The model we use (`all-MiniLM-L6-v2`) converts every piece of text into a list of **384 numbers** — each number representing a different "dimension of meaning."

```python
'students learn better with practice'
→  [0.12, -0.34, 0.87, 0.02, -0.55, 0.23, ...  384 numbers total]

'active learning improves retention'
→  [0.11, -0.31, 0.85, 0.04, -0.52, 0.25, ...  384 numbers total]
                                                  ↑
                          Notice: similar meaning → similar numbers
```

No single number means anything on its own — it's the *pattern* across all 384 that encodes the meaning. Think of it like a colour: a single RGB value doesn't describe a colour, but R=255, G=165, B=0 together say "orange" precisely.

---

## How "closeness" is measured

Once we have two vectors, we measure how similar they are using **cosine similarity**:

- **1.0** = identical meaning
- **0.8+** = very similar (paraphrases, same concept)
- **0.5–0.7** = related topics
- **below 0.3** = unrelated

This is the "relevance score" you'll see when the system retrieves chunks. It tells you how confidently the retrieval system thinks a chunk answers your question — *before* Granite even reads it.

---

## Why does this matter for architects?

When NotebookLM gives your student a wrong citation, it's often because the *retrieval step failed* — the question and the correct passage ended up far apart on the meaning map. This happens when:

- The student's vocabulary is very different from the document's vocabulary
- The question is too vague to land near any specific passage
- The document uses field-specific jargon the model doesn't map well

Understanding this lets you design better student prompts and better document collections.

### 👀 See Embeddings in Action

Before embedding your actual documents, let's build intuition with a small demonstration. Run the cell below — it embeds six short phrases and shows you the actual numbers, then computes similarity scores so you can see the meaning-closeness in action.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the embedding model
print('Loading embedding model...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
dim = embed_model.get_sentence_embedding_dimension()
print(f'✅ Model loaded — converts any text to {dim} numbers\n')

# ── DEMONSTRATION: six phrases to compare ────────────────────────────────
demo_phrases = [
    'students learn better with active practice',
    'active learning improves student retention',
    'exam scores correlate with attendance',
    'interest rates affect bond prices',
    'quarterly earnings disappointed investors',
    'DNA replication is essential for cell division',
]

demo_vectors = embed_model.encode(demo_phrases)

# Show the actual numbers for the first phrase
print('WHAT AN EMBEDDING LOOKS LIKE:')
print(f'   Phrase: "{demo_phrases[0]}"')
print(f'   Vector (first 10 of {dim} numbers):')
print(f'   {demo_vectors[0][:10].round(3).tolist()}')
print(f'   ... and {dim - 10} more numbers')
print(f'   The full pattern of {dim} numbers encodes the meaning.\n')

# Compute cosine similarities between all pairs
from sklearn.metrics.pairwise import cosine_similarity
sims = cosine_similarity(demo_vectors)

print('SIMILARITY SCORES (1.0 = identical meaning, 0.0 = unrelated):')
print(f'{'─'*60}')

# Show most interesting comparisons
comparisons = [
    (0, 1, 'Both about active learning / student retention'),
    (0, 2, 'Both about students, but different concepts'),
    (0, 3, 'Education vs Finance — very different'),
    (3, 4, 'Both about finance'),
    (0, 5, 'Education vs Biology'),
]
for i, j, label in comparisons:
    score = sims[i][j]
    bar = '█' * int(score * 20)
    print(f'  {score:.3f}  {bar}')
    print(f'         "{demo_phrases[i][:45]}"')
    print(f'         "{demo_phrases[j][:45]}"')
    print(f'         → {label}\n')


### 🗺️ The Meaning Map — Visualised

The similarity scores above are computed in 384-dimensional space — impossible to visualise directly. But we can use a technique called **PCA** (Principal Component Analysis) to compress those 384 dimensions down to 2, and plot them on a regular chart.

Think of it like viewing a 3D sculpture from directly above — you lose some information, but you can see the overall structure of what's near what.

**What to look for in the plot:**
- Similar phrases should cluster together
- Finance phrases should land far from education phrases
- The ★ questions should appear near the phrases that could answer them

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Add some questions to the mix
questions = [
    'How does practice affect learning?',
    'What drives stock market changes?',
]
all_phrases = demo_phrases + questions
all_vectors = embed_model.encode(all_phrases)

# Compress to 2D with PCA
pca = PCA(n_components=2)
coords = pca.fit_transform(all_vectors)

# Colour-code by topic
colors = ['#1f77b4','#1f77b4','#1f77b4',  # blue = education
          '#d62728','#d62728',            # red = finance
          '#2ca02c',                       # green = biology
          '#ff7f0e','#ff7f0e']             # orange = questions
markers = ['s','s','s','s','s','s','*','*']
sizes   = [160,160,160,160,160,160,300,300]

fig, ax = plt.subplots(figsize=(11, 7))

for i, phrase in enumerate(all_phrases):
    ax.scatter(coords[i,0], coords[i,1], c=colors[i], marker=markers[i],
               s=sizes[i], zorder=5, edgecolors='white', linewidths=0.5)
    label = f'★ Q: "{phrase[:40]}"' if i >= len(demo_phrases) else f'"{phrase[:42]}"'
    offset = (8, 8) if i % 2 == 0 else (8, -16)
    ax.annotate(label, (coords[i,0], coords[i,1]), fontsize=8.5,
                xytext=offset, textcoords='offset points',
                color='#ff7f0e' if i >= len(demo_phrases) else 'black')

# Draw dashed lines from each question to its nearest phrase
for qi in range(len(demo_phrases), len(all_phrases)):
    dists = [np.linalg.norm(coords[qi] - coords[pi]) for pi in range(len(demo_phrases))]
    nearest = np.argmin(dists)
    ax.plot([coords[qi,0], coords[nearest,0]], [coords[qi,1], coords[nearest,1]],
            '--', color='#ff7f0e', alpha=0.5, linewidth=1.5)

# Legend
legend_elements = [
    mpatches.Patch(color='#1f77b4', label='Education phrases'),
    mpatches.Patch(color='#d62728', label='Finance phrases'),
    mpatches.Patch(color='#2ca02c', label='Biology phrase'),
    mpatches.Patch(color='#ff7f0e', label='★ Your questions'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

ax.set_title('The Meaning Map — How Embeddings Organise Text by Concept\n'
             '(dashed lines show: which phrase is each question closest to?)',
             fontsize=12)
ax.set_xlabel('Meaning Dimension 1 (compressed from 384 dimensions)')
ax.set_ylabel('Meaning Dimension 2')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print('Notice:')
print('  · Education phrases cluster together (blue squares)')
print('  · Finance phrases cluster together (red squares)')
print('  · The orange questions land near the phrases that can answer them')
print('  · This spatial relationship is what RAG retrieval is doing —')
print('    finding your question\'s nearest neighbors on this map.')


### 📦 Now Embed Your Actual Documents

The demo above used 6 short phrases. Now we run the same process on all the chunks from your parsed PDFs — potentially hundreds of passages.

> ⏱️ **1–3 minutes** depending on how many chunks you have.

In [ ]:
import chromadb

# Embed all document chunks
print(f'Embedding {len(all_chunks)} chunks...')
chunk_vectors = embed_model.encode(all_chunks, show_progress_bar=True)
print(f'✅ Embeddings created — shape: {chunk_vectors.shape}')
print(f'   ({chunk_vectors.shape[0]} chunks × {chunk_vectors.shape[1]} dimensions each)\n')

# Store in ChromaDB
client = chromadb.Client()
try:
    client.delete_collection('my_documents')
except:
    pass

collection = client.create_collection(
    name='my_documents',
    metadata={'hnsw:space': 'cosine'}
)
collection.add(
    documents=all_chunks,
    metadatas=all_metadata,
    ids=all_ids,
    embeddings=chunk_vectors.tolist()
)
print(f'✅ Vector index built — {collection.count()} chunks stored and searchable.')
print(f'   ChromaDB is now holding the meaning map of your documents.')
print(f'   Every question you ask gets placed on this map and finds its nearest chunks.')


---
# 🤖 Step 4: Load IBM Granite

Now we load Granite 3.1 2B — the same model from Tutorial 1. This is the **generation** step: Granite reads the retrieved chunks and writes a grounded answer.

**Granite's role here is specific and limited:**  
It does *not* search your documents. It only sees what the retrieval step gives it. This is what "grounded" means — Granite can only use the text in the context window, not its general training knowledge. If the answer isn't in the retrieved chunks, a well-behaved Granite response will say so.

🔗 [IBM Granite on Hugging Face](https://huggingface.co/ibm-granite/granite-3.1-2b-instruct) · [IBM Granite GitHub](https://github.com/ibm-granite)

> ⏱️ **1–2 minutes** on first run (downloads ~1.5 GB from Hugging Face).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = 'ibm-granite/granite-3.1-2b-instruct'

print(f'Loading: {MODEL_NAME}')
print('Source: https://huggingface.co/ibm-granite/granite-3.1-2b-instruct')
print('This takes 1–2 minutes on first run...\n')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)

param_count = sum(p.numel() for p in model.parameters()) / 1e9
device = next(model.parameters()).device
print(f'✅ IBM Granite 3.1 loaded')
print(f'   Parameters : {param_count:.1f} billion')
print(f'   Running on : {device}')
print(f'   Model page : https://huggingface.co/{MODEL_NAME}')


### The `ask_documents()` Function

This function wires together the full RAG pipeline:

```
Your question
    │
    ▼
embed_model.encode()       ← converts question to a vector
    │
    ▼
collection.query()         ← finds the n closest chunks on the meaning map
    │
    ▼
Build grounded prompt      ← [question + retrieved chunks]
    │
    ▼
IBM Granite                ← generates a cited answer from that prompt only
    │
    ▼
Answer + source citations
```

Run this cell to define the function — it produces no output yet.

In [ ]:
def ask_documents(question, n_chunks=3, max_new_tokens=500, temperature=0.3,
                  show_sources=True, show_prompt=False):
    """
    Ask a question about your uploaded documents.

    Parameters
    ----------
    question       : str   — your question in plain English
    n_chunks       : int   — how many chunks to retrieve (default 3)
    max_new_tokens : int   — max response length (default 500 ≈ one page)
    temperature    : float — 0.1 precise / 0.3 default / 0.7 creative
    show_sources   : bool  — print retrieved source chunks (default True)
    show_prompt    : bool  — print the full RAG prompt sent to Granite
    """
    # ── 1. Embed the question ─────────────────────────────────────────────
    q_vector = embed_model.encode([question]).tolist()

    # ── 2. Retrieve nearest chunks ────────────────────────────────────────
    results = collection.query(query_embeddings=q_vector, n_results=n_chunks)

    # ── 3. Build grounded RAG prompt ──────────────────────────────────────
    context = ''
    for i in range(n_chunks):
        source  = results['metadatas'][0][i]['source']
        text    = results['documents'][0][i]
        sim     = round(1 - results['distances'][0][i], 3)
        context += f'[{i+1}] {source} (relevance: {sim}):\n{text}\n\n'

    prompt = (
        'You are a research assistant. Answer the question using ONLY the provided '
        'context. Cite sources using [1], [2], etc.\n'
        'If the context does not contain enough information, say so clearly — '
        'do not guess or add information from outside the context.\n\n'
        f'CONTEXT:\n{context}'
        f'QUESTION: {question}\n'
        f'ANSWER:'
    )

    if show_prompt:
        print('─── RAG PROMPT SENT TO GRANITE ───')
        print(prompt[:1200], '...' if len(prompt) > 1200 else '')
        print('─' * 35 + '\n')

    # ── 4. Generate answer with IBM Granite ───────────────────────────────
    messages = [{'role': 'user', 'content': prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True, top_p=0.9
        )
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    ).strip()

    # ── 5. Display ────────────────────────────────────────────────────────
    print(f'❓ QUESTION: {question}')
    print('─' * 65)
    print('🤖 GRANITE\'S ANSWER:')
    print(response)

    if show_sources:
        print()
        print('📚 RETRIEVED SOURCES (what Granite was given):')
        for i in range(n_chunks):
            source  = results['metadatas'][0][i]['source']
            sim     = round(1 - results['distances'][0][i], 3)
            chunk_i = results['metadatas'][0][i]['chunk_index']
            preview = results['documents'][0][i][:120].replace('\n', ' ')
            flag = '✅' if sim > 0.5 else ('⚠️ ' if sim > 0.3 else '❌')
            print(f'  {flag} [{i+1}] {source}  chunk {chunk_i}  (relevance: {sim})')
            print(f'       "{preview}..."')

    return response

print('✅ ask_documents() is ready.')


---
# Step 5: Ask Questions About Your Documents

Everything is set up. We'll run the same question two ways so you can see the difference.

**Reading the output:**
- Citations `[1]`, `[2]` in the answer refer to the Retrieved Sources below it
- Relevance scores: ✅ > 0.5 (strong match) · ⚠️ 0.3–0.5 (weak) · ❌ < 0.3 (poor)
- If all relevance scores are ❌, the answer probably isn't in your documents

---

### 🔹 First: Try the bare question as written

Run this exactly as-is. Notice the relevance scores and how specific the answer is.

---

### 🔹 Now try the same question using the ICI framework

The bare question above lands vaguely on the meaning map — it could mean anything. A structured ICI prompt gives the embedding model more to work with, landing closer to the specific passages that can actually answer it.

**Compare:**
- Are the relevance scores higher?
- Is the answer more specific and useful?
- Did different chunks get retrieved?

> **This is the same ICI skill from Day 2** — Context → Instruction → Input. The prompt recipe isn't just for chatbots. It applies to any AI interaction, including querying a RAG system.

In [ ]:
# ── Change this question to something relevant to your documents ────────
ask_documents("What are the main findings of these papers?")


### Ask Your Own Questions

Replace the question below and re-run. Try it twice — once as a bare question, then rewrite it using the ICI framework and compare the relevance scores and answer quality.

**Ideas to try:**
- Something specific: *"What sample sizes were used?"*
- A comparison: *"Do any papers contradict each other?"*
- Something your documents don't cover — notice what Granite does

**ICI reminder — same framework from Day 2:**
- **Context:** Set a role and audience — *"You are a research assistant supporting a lit review..."*
- **Instruction:** Tell it exactly what to do and how to format — *"Summarize as 3 bullet points for a faculty audience"*
- **Input:** The specific question or focus — *"...about the impact of AI on student outcomes"*

> **Architect's experiment:** Add `show_prompt=True` to see the exact prompt Granite receives — this is the Tutorial 2 Step 6 moment with your real data.

In [ ]:
my_question = "What methodologies do these papers use?"

# Add show_prompt=True to see the full RAG prompt sent to Granite
ask_documents(my_question, n_chunks=3)


### 🔬 Experiment: Context Window Size

The `n_chunks` parameter controls how much context Granite receives. This is an architectural decision you make as the instructor when designing a RAG system for students.

Run both cells and compare the answers. Think about:
- Does more context always mean a better answer?
- At what point does extra context introduce noise?
- How would you explain this trade-off to a student who gets a vague answer?

In [ ]:
comparison_q = "What are the key conclusions?"

print('━'*65)
print('1 CHUNK — minimal context (Granite sees only the top result)')
print('━'*65)
ask_documents(comparison_q, n_chunks=1, show_sources=False)

print()
print('━'*65)
print('5 CHUNKS — more context (Granite sees five passages)')
print('━'*65)
ask_documents(comparison_q, n_chunks=5, show_sources=False)


---
# 🔗 Connecting Back to Tutorial 2

Now that you've run a real pipeline, the Tutorial 2 failure modes should feel concrete:

| Tutorial 2 failure | What it looks like here |
|---|---|
| **Ingest failure** | Docling returns near-empty text — scanned or image-only PDF |
| **Chunk boundary problem** | Answer splits across two chunks, neither complete on its own |
| **Vocabulary mismatch** | Your question uses different words — relevance scores all ❌ |
| **Retrieval failure** | Correct passage exists but lands far from the question on the map |
| **Confident wrong answer** | High relevance score, but Granite misread or paraphrased badly |

**The architect insight:** When a student gets a bad answer from NotebookLM, you can now diagnose *which step in the pipeline* likely failed — and design your document collection and student prompts to reduce those specific failures.

---
# 📁 Use Your Own Folder

The cell below is a clean template. Replace `FOLDER_PATH`, re-run from the **Parse** step, and you have a RAG system over your own documents.

**Granite stays loaded — you don't need to re-run Step 4.**  
Just re-run Steps 1 → 2 → 3 with the new folder, then `ask_documents()` will query the new collection.

**Good document collections to try:**
- Research papers for a current lit review
- Course syllabi from across your department
- Accreditation or assessment reports
- Your own draft manuscripts — they never leave this Colab session

In [ ]:
# ── YOUR FOLDER — replace this and re-run Steps 1–3 ────────────────────────
FOLDER_PATH = '/content/drive/MyDrive/[YOUR FOLDER NAME HERE]'

# Steps to re-run after changing FOLDER_PATH:
#   1. The Docling parse cell   (Step 1)
#   2. The chunk cell           (Step 2)
#   3. The embed + store cell   (Step 3)
#   Granite stays loaded — skip Step 4
#   Then ask_documents() queries your new collection

# To find your path:
#   1. Open Google Drive in your browser
#   2. Navigate to your folder
#   3. Build path as: /content/drive/MyDrive/FolderName/SubfolderName
#   Note: 'MyDrive' not 'My Drive' (no space)


---
# Tutorial 3 Wrap-Up

**What you built:**

| Step | IBM Tool | What happened |
|---|---|---|
| Parse | **Docling** | PDFs → clean structured text, layout-aware |
| Chunk | Python | Text → ~400-word searchable pieces |
| Embed | sentence-transformers | Chunks → 384-dimensional meaning vectors |
| Store | ChromaDB | Vectors → searchable meaning map |
| Retrieve | ChromaDB | Question → nearest chunks on the map |
| Generate | **IBM Granite 3.1** | Retrieved chunks → cited answer |

**The difference from NotebookLM:**  
You controlled every step — chunk size, embedding model, number of retrieved chunks, which LLM generates. That control is what being an architect at the Embedded AI level means.

**IBM open-source tools used:**
- 🔗 [Docling](https://github.com/DS4SD/docling) — PDF parsing
- 🔗 [IBM Granite 3.1 2B](https://huggingface.co/ibm-granite/granite-3.1-2b-instruct) — Answer generation

**What's next — Day 4: Agentic AI**

> Today's pipeline answered questions when *you* asked them.  
> Tomorrow's AI decides *what to search, when to act, and how to chain steps together* — without you driving each step manually.

---
*© 2026 Dr. Jeanne McClure · CC BY-NC 4.0 · NSF DUE-2313644*